# Throughput: torchcrop on CPU and GPU, SIMPLACE on CPU

**Unit: cell-years per second.** One cell simulated through one season. It is
the only unit both models share: SIMPLACE runs a project line (one cell, every
season from 1979 to 2024) per JVM thread, and torchcrop runs a `[B, T]` batch of
cell-seasons per forward pass.

| Source | How it is measured | Device |
|---|---|---|
| torchcrop | `cm4eu benchmark` times real calibration batches: cached pool seasons, real soil and site, through `Calibrator.batch_score` (*forward*) and `batch_gradients` (*fwd+bwd*, checkpointed). Minimum of the repeats after one warm-up; GPU timers synchronise first. | CPU, GPU |
| SIMPLACE | Not re-run. Each SLURM task logs `OK lines A-B in XmYs`, and lines × seasons ÷ seconds ÷ JVMs is the per-core rate. | CPU only (Java) |

SIMPLACE cannot run on a GPU, so it appears as a **per-core band**, not a
curve. torchcrop's CPU throughput is also per process: its day loop is a Python
loop, so threads add nothing (measured flat at 1, 4, 10 and 20) and only
processes scale. That makes "per core" the fair comparison on CPU, and a whole
GPU against one core the honest framing on the GPU side.

## 0. Setup

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cropmodelling4eu.evaluation.style import (
    DIVERGING_CMAP, SEQUENTIAL_CMAP, diverging_norm, use_style,
)

logging.basicConfig(level=logging.WARNING, force=True)
P = use_style("light")
RUN = Path("/data01/FDS/muduchuru/Data/SIMPLACE/cropmodelling4eu/winter_wheat_2000_2024_euval")

#: `cm4eu benchmark --output ... [--append]` from the CPU and the GPU job.
BENCH = RUN / "benchmark" / "benchmark.parquet"
#: Per-task SIMPLACE timings the same command wrote beside it.
TASKS = BENCH.with_name(BENCH.stem + "_simplace_tasks.parquet")
#: Cell-years one calibration epoch simulates: pool rows x data.fraction.
#: Read off the run's log ("pool: N (unit, year, cell) rows"); EU_val phenology.
EPOCH_CELL_YEARS = int(73_562 * 0.20)
#: Forked region workers on the CPU path (CAL_WORKERS).
CPU_WORKERS = 29

bench = pd.read_parquet(BENCH)
tasks = pd.read_parquet(TASKS) if TASKS.is_file() else pd.DataFrame()
torchcrop = bench[bench["model"] == "torchcrop"].copy()
simplace = bench[bench["model"] == "simplace"].copy()

#: Colour follows the entity, never its rank: fixed slots, never cycled.
COLOUR = {"torchcrop cpu": P["series_1"], "torchcrop cuda": P["series_2"],
          "simplace": P["series_3"]}
bench.groupby(["model", "label", "device", "mode"]).size().rename("rows").to_frame()

## 1. Throughput against batch size

Solid lines are mixed-region batches; dashed lines, where present, are the
region-packed path, whose batch the largest region caps. The grey band is
SIMPLACE's per-core rate across every logged task (5th to 95th percentile),
with the median line.

In [ ]:
modes = [m for m in ("forward", "fwd+bwd") if m in set(torchcrop["mode"])]
fig, axes = plt.subplots(1, len(modes), figsize=(5.2 * len(modes), 3.8), sharey=True,
                         squeeze=False)
for ax, mode in zip(axes[0], modes):
    block = torchcrop[torchcrop["mode"] == mode]
    for (device, label), line in block.groupby(["device", "label"]):
        line = line.sort_values("batch_size")
        key = f"torchcrop {device}"
        ax.plot(line["batch_size"], line["cell_years_per_s"], marker="o", ms=4, lw=2,
                ls="-" if label == "mixed" else "--", color=COLOUR[key],
                label=f"{key} ({label})")
        last = line.iloc[-1]
        ax.annotate(f"{last['cell_years_per_s']:,.0f}", (last["batch_size"], last["cell_years_per_s"]),
                    xytext=(4, 0), textcoords="offset points", va="center",
                    color=P["secondary"], fontsize=8)
    if not tasks.empty:
        lo, mid, hi = tasks["cell_years_per_core_s"].quantile([0.05, 0.5, 0.95])
        ax.axhspan(lo, hi, color=P["grid"], zorder=0, label="SIMPLACE per core (p5-p95)")
        ax.axhline(mid, color=COLOUR["simplace"], lw=2, label=f"SIMPLACE median {mid:.1f}")
    ax.set(xscale="log", yscale="log", xlabel="batch size [cell-years]", title=mode)
axes[0][0].set_ylabel("cell-years per second")
axes[0][-1].legend(loc="lower right", frameon=False, fontsize=8)
fig.tight_layout()

## 2. Peak memory

On the GPU this is `torch.cuda.max_memory_allocated`, reset per timing. On the
CPU it is the process's peak RSS, a high-water mark that only rises across a
sweep. So the CPU line is an upper bound, and the sweep runs in ascending order
for that reason.

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.6))
for (device, mode), line in torchcrop[torchcrop["label"] == "mixed"].groupby(["device", "mode"]):
    if mode == "loader":
        continue
    line = line.sort_values("batch_size")
    ax.plot(line["batch_size"], line["peak_mem_gb"], marker="o", ms=4, lw=2,
            ls="-" if mode == "fwd+bwd" else ":", color=COLOUR[f"torchcrop {device}"],
            label=f"{device} {mode}")
ax.set(xscale="log", xlabel="batch size [cell-years]", ylabel="peak memory [GB]")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()

## 3. What one calibration epoch costs

This is an estimate, not a measurement: `EPOCH_CELL_YEARS` divided by each
path's best fwd+bwd throughput. The CPU path is the forked pool: one process per
region, `CPU_WORKERS` at once, each at the region-packed per-process rate. The
GPU path is one process running mixed batches. Validation is excluded on both
sides, as it scales the same way. The forked bar needs the region-packed CPU
sweep (`--region-packed`), which `submit_benchmark.sh` runs.

In [ ]:
step = torchcrop[torchcrop["mode"] == "fwd+bwd"]
best = step.groupby(["device", "label"])["cell_years_per_s"].max()
paths = {}
# The fork pool runs region-packed batches, so it is scaled from that rate only;
# multiplying the mixed rate by the worker count would credit it batches it
# cannot form.
if ("cpu", "region-packed") in best:
    paths[f"CPU, {CPU_WORKERS} forked workers, region-packed"] = (
        EPOCH_CELL_YEARS / (best["cpu", "region-packed"] * CPU_WORKERS))
if ("cpu", "mixed") in best:
    paths["CPU, 1 process, mixed"] = EPOCH_CELL_YEARS / best["cpu", "mixed"]
if ("cuda", "mixed") in best:
    paths["GPU, 1 process, mixed"] = EPOCH_CELL_YEARS / best["cuda", "mixed"]

epoch = pd.Series(paths, name="minutes").div(60).sort_values()
fig, ax = plt.subplots(figsize=(5.6, 0.5 + 0.45 * len(epoch)))
colours = [COLOUR["torchcrop cuda"] if "GPU" in k else COLOUR["torchcrop cpu"] for k in epoch.index]
ax.barh(epoch.index, epoch.values, color=colours, height=0.6)
for y, v in enumerate(epoch.values):
    ax.text(v, y, f"  {v:.1f} min", va="center", color=P["secondary"], fontsize=8)
ax.set(xlabel=f"minutes per epoch ({EPOCH_CELL_YEARS:,} cell-years, fwd+bwd)")
ax.invert_yaxis()
fig.tight_layout()
epoch.to_frame()

## 4. How much of a step is the data loader

A cache gather plus collate (`loader`) as a share of a full fwd+bwd step. A
share that grows with B means the memmap read, not the model, is becoming the
bottleneck. The fix there is pinned host memory or a prefetch thread, not a
bigger GPU.

In [ ]:
pivot = torchcrop.pivot_table(index=["device", "label", "batch_size"], columns="mode",
                              values="seconds", aggfunc="min")
if {"loader", "fwd+bwd"} <= set(pivot.columns):
    share = (pivot["loader"] / pivot["fwd+bwd"]).rename("loader share").reset_index()
    fig, ax = plt.subplots(figsize=(5.6, 3.4))
    for (device, label), line in share.groupby(["device", "label"]):
        ax.plot(line["batch_size"], 100 * line["loader share"], marker="o", ms=4, lw=2,
                ls="-" if label == "mixed" else "--", color=COLOUR[f"torchcrop {device}"],
                label=f"{device} ({label})")
    ax.set(xscale="log", xlabel="batch size [cell-years]", ylabel="loader / step [%]")
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()

## 5. SIMPLACE's spread across tasks

Task ranges are contiguous in SimplaceID order, which runs north to south. So
the per-task rate carries a latitude signal, which a single median hides.

In [ ]:
if not tasks.empty:
    fig, ax = plt.subplots(figsize=(5.6, 3.4))
    for i, (source, block) in enumerate(tasks.groupby("source")):
        ax.hist(block["cell_years_per_core_s"], bins=40, histtype="step", lw=2,
                color=P[f"series_{3 + i % 6}"], label=Path(source).parent.name
                if Path(source).name == "simplace" else Path(source).name)
    ax.set(xlabel="cell-years per second per core", ylabel="tasks")
    ax.legend(frameon=False, fontsize=7)
    fig.tight_layout()

## 6. Table view

In [ ]:
bench.sort_values(["model", "device", "label", "mode", "batch_size"]).round(3)